# 📊 FinDataMining
Notebook 01: **Extracción de Datos**

---

## Preliminares

* Importar librerías:

In [1]:
import pandas as pd
from src.config import market_index_file

%load_ext autoreload
%autoreload 2
from src.extract import *

## Extracción de datos financieros de `simFin`

Luego de haber creado la cuenta gratuita e ingresado la clave API en el repositorio local (ver [README.md](README.md)), se obtienen de `simFin` los datos financieros históricos de un período máximo de 5 años, con un lag de 1 año. A partir de los datos descargados, se genera la lista de tickers en el universo del proyecto, según los criterios:

- Umbral de filas: cantidad mínima de reportes trimestrales disponibles (por defecto 19).

- Columna ranking: se calcula el promedio de la columna indicada (por defecto TotalRevenue).

- Cantidad de tickers: luego de rankear según la columna indicada, se toman los top n tickers (por defecto 550).

In [2]:
# Extraer datos de simFin
df_simfin_unfiltered = extraer_simfin()

# Genera el universo inicial de tickers según los criterios establecidos
tickers_universe_list = generar_universo_tickers(df_simfin_unfiltered)

# Filtra los tickers del universo
df_simfin_filtered = df_simfin_unfiltered[df_simfin_unfiltered['Ticker'].isin(tickers_universe_list)]

# Estandarizar columnas y fechas para que coincidan con el formato de yfinance
df_simfin_standarized = estandarizar_simfin(df_simfin_filtered)
print("Datos extraidos de simFin, dimensiones:", df_simfin_standarized.shape)

Dataset "us-income-quarterly" on disk (46 days old).
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (46 days old).
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (46 days old).
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-income-banks-quarterly" on disk (45 days old).
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-balance-banks-quarterly" on disk (45 days old).
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-cashflow-banks-quarterly" on disk (45 days old).
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-income-insurance-quarterly" on disk (45 days old).
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from d

## Extracción de precios de `yfinance`

Se Extraen precios mensuales de los tickers y del índice SPY. El formato de fechas obtenido es el día primero de cada mes, indicando el inicio del mes que se informa. Por lo tanto, los precios alineados con dichas fechas son los precios **Open**.

In [3]:
# Si existe un fichero raw_data guardado, se verifican los datos que requieran actualización
new_data_tickers, df_simfin_new_data = gestionar_actualizacion(tickers_universe_list, df_simfin_standarized)

# Extraer precios y obtener lista de tickers sin precios
df_prices, tickers_sin_precios = extraer_precios(new_data_tickers)

# Se extrae precio del Índice de Mercado para usar en cálculos y se guarda en fichero 
df_index, _ = extraer_precios(['SPY'])

# Asegurar que el directorio exista antes de guardar los datos
market_index_file.parent.mkdir(parents=True, exist_ok=True)

df_index.to_parquet(market_index_file)

tickers_con_precios_nuevos = df_prices['Ticker'].unique().tolist()

print(f"Precios extraídos para {len(tickers_con_precios_nuevos)} tickers.")

Iniciando descarga masiva para 550 tickers únicos...


[*********************100%***********************]  550 of 550 completed

2 Failed downloads:
['SCHN', 'TPH']: YFPricesMissingError('possibly delisted; no price data found  (period=6y) (Yahoo error = "No data found, symbol may be delisted")')


Reestructurando los datos...
Advertencia: No se encontraron datos para 2 tickers.
Extracción completada.
Iniciando descarga masiva para 1 tickers únicos...


[*********************100%***********************]  1 of 1 completed

Reestructurando los datos...
Extracción completada.
Precios extraídos para 548 tickers.


In [4]:
df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39099 entries, 0 to 39098
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    39099 non-null  datetime64[ns]
 1   Ticker  39099 non-null  object        
 2   Close   39099 non-null  float64       
 3   Open    39099 non-null  float64       
 4   Volume  39099 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 1.5+ MB


## Extracción de datos complementarios

Se obtiene la lista de tickers del S&P 500 desde el fichero `constituents.csv`:

* De dicho fichero se obtiene para los tickers que pertenecen al índice, la fecha en la que fueron agregados con la columna `DataAdded`.
* Si no existe el fichero, lo descarga desde GitHub.
* Para actualizar la lista de componentes, cambiar `force_update` a True, ejecutar y volver a dejar en False.

In [5]:
# Obtener los componentes del índice S&P 500
ruta_sp500 = descargar_constituents_sp(force_update=False) 

# Cargar y limpiar datos
print("Extrayendo información complementaria")
df_tickers_sp_raw = pd.read_csv(ruta_sp500)
df_tickers_sp = limpiar_constituents_sp(df_tickers_sp_raw)

Usando archivo constituents.csv local.
Extrayendo información complementaria


* Extraer de `yfinance` la información de sector e industria (demora unos minutos):

In [6]:
# Extraer info de Sector e Industry de yfinance
df_info = extraer_info(tickers_con_precios_nuevos)
df_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 548 entries, 0 to 547
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Ticker    548 non-null    object
 1   Sector    548 non-null    object
 2   Industry  548 non-null    object
dtypes: object(3)
memory usage: 13.0+ KB


* Se unen los datasets de info complementaria:

In [7]:
# Se unen los datos complementarios
df_supplementary = pd.merge(
    df_info,
    df_tickers_sp,
    on= 'Ticker',
    how= 'left'
)
print("Unidos los datos complementarios. Dimensiones:", df_supplementary.shape)

Unidos los datos complementarios. Dimensiones: (548, 4)


## Extracción de  datos financieros de `yfinance`

Se descargan los últimos 4 reportes trimestrales.

`yfinance` no incluye la fecha de publicación, sólo la fecha de los Estados Financieros. Para evitar el "LookAhead" bias, se ofrecen 2 alternativas con el parametro `aproximar_fechas`:

*aproximar_fechas = True*: 
* Se estima la fecha de publicación con una regla de 30 días por defecto (editable en src/config.py). 
* *Demora de primera ejecución* de aproximadamente 10 minutos.

*aproximar_fechas = False*: 
* Obtiene la fecha real de publicación usando `yf.earnings_dates`. 
* Se aplica la regla de 30 días sólo si no se encuentra. 
* La primera vez en ejecutar puede demorar 30 minutos o más.

In [8]:
print("Extrayendo datos financieros de yfinance, puede demorar varios minutos.")
df_yfinance, tickers_sin_financials = extraer_financials(
    tickers_con_precios_nuevos, 
    aproximar_fechas = False
    )
print("Dimensiones:", df_yfinance.shape)

Extrayendo datos financieros de yfinance, puede demorar varios minutos.
Procesando 125/548: CTA-PB...    

CTA-PB: No earnings dates found, symbol may be delisted


Aviso: Datos de ganancias vacíos para CTA-PB. Usando estimación.
Procesando 412/548: RSTRF...    

RSTRF: No earnings dates found, symbol may be delisted


Aviso: Datos de ganancias vacíos para RSTRF. Usando estimación.
Procesando 449/548: TAP...      Aviso: Error crítico obteniendo fechas para TAP. Usando estimación. (Failed to perform, curl: (28) Operation timed out after 30012 milliseconds with 43681 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.)
Procesando 450/548: TDG...    Sin datos financieros trimestrales para TDG
Procesando 545/548: SATS...    

SATS: No earnings dates found, symbol may be delisted


Aviso: Datos de ganancias vacíos para SATS. Usando estimación.
Procesando 548/548: TMHC...    
¡Extracción finalizada!
Dimensiones: (2188, 25)


In [9]:
df_yfinance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2188 entries, 0 to 2187
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Date                           2188 non-null   datetime64[ns]
 1   Total Revenue                  2116 non-null   float64       
 2   Gross Profit                   1989 non-null   float64       
 3   Operating Income               2003 non-null   float64       
 4   Net Income                     2116 non-null   float64       
 5   EBITDA                         2003 non-null   float64       
 6   Basic Average Shares           2141 non-null   float64       
 7   Cash And Cash Equivalents      2110 non-null   float64       
 8   Current Debt                   1581 non-null   float64       
 9   Long Term Debt                 1979 non-null   float64       
 10  Total Debt                     2095 non-null   float64       
 11  Stockholders Equi

## Unión de datasets y almacenamiento

* Se unen los datos financieros de `simFin` y `yfinance`, quitando espacios en los nombres de las variables financieras:

In [10]:
# Se guardan los tickers sin datos en fichero .csv para descartarlos si se itera la extracción
sin_datos_set = set(tickers_sin_precios+tickers_sin_financials)
guardar_tickers_sin_datos(sin_datos_set)

df_financials_completo = unir_financials(df_yfinance, df_simfin_new_data)
print("Unidos datasets de financials. Dimensiones:", df_financials_completo.shape)

Se han encontrado 8 filas con Ticker y Date solapados (presentes en ambas fuentes).
Unidos datasets de financials. Dimensiones: (2502, 25)


In [11]:
df_financials_completo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2502 entries, 0 to 2573
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Ticker                       2502 non-null   object        
 1   Date                         2502 non-null   datetime64[ns]
 2   BasicAverageShares           2456 non-null   float64       
 3   CapitalExpenditure           2377 non-null   float64       
 4   CashAndCashEquivalents       2496 non-null   float64       
 5   CurrentAssets                2349 non-null   float64       
 6   CurrentDebt                  1845 non-null   float64       
 7   CurrentLiabilities           2349 non-null   float64       
 8   DepreciationAndAmortization  2112 non-null   float64       
 9   EBITDA                       2003 non-null   float64       
 10  FinancialsSource             2502 non-null   object        
 11  FinancingCashFlow            2499 non-null   flo

Se convierten las variables financieras de flujo (Income Statement y Cash Flow) a a su equivalente anualizado TTM (Total Twelve Months), añadiendo el sufijo "_TTM".

Esta transformación se ejecuta **antes** de cruzar los datos financieros trimestrales con los precios mensuales para garantizar la integridad matemática de los cálculos:

1. **Prevención de Distorsión por Frecuencia:** Si se aplicara un *forward fill* mensual antes del TTM, cada valor trimestral se repetiría 3 veces. Al aplicar una suma móvil de 12 meses sobre esos datos, estaríamos sumando cada trimestre 3 veces, inflando el valor final en un 300%.
2. **Tratamiento Diferenciado (Flujo vs. Foto):** Las variables de flujo exigen calcularse en su temporalidad original (sumando estrictamente 4 trimestres reales). Una vez consolidado el TTM anualizado, este se convierte estructuralmente en una variable "de foto", haciendo seguro el *forward fill* en la fase mensual para propagar la última información pública disponible sin alterar el acumulado.

In [12]:
df_financials_ttm = transformar_flujos_a_ttm(df_financials_completo)
df_financials_ttm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2502 entries, 0 to 2573
Data columns (total 25 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ticker                           2502 non-null   object        
 1   Date                             2502 non-null   datetime64[ns]
 2   CashAndCashEquivalents           2496 non-null   float64       
 3   CurrentAssets                    2349 non-null   float64       
 4   CurrentDebt                      1845 non-null   float64       
 5   CurrentLiabilities               2349 non-null   float64       
 6   FinancialsSource                 2502 non-null   object        
 7   LongTermDebt                     2344 non-null   float64       
 8   StockholdersEquity               2494 non-null   float64       
 9   TotalAssets                      2500 non-null   float64       
 10  TotalDebt                        2095 non-null   float64       
 

Se une el dataframe de precios con el de datos financieros completo. Ya que los precios tienen una granularidad mensual y los Estados Financieros son trimestrales, el *left join* producirá nulos para los valores financieros en las fechas intermedias. Los mismos serán tratados con ffill en la limpieza final:

In [13]:
# Unir precios con datos financieros
df_merged = pd.merge(
    df_prices, 
    df_financials_ttm, 
    on=['Date', 'Ticker'],
    how='left' 
)
print("Unidos datasets de precios y financials. Dimensiones:", df_merged.shape)

Unidos datasets de precios y financials. Dimensiones: (39099, 28)


In [14]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39099 entries, 0 to 39098
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Date                             39099 non-null  datetime64[ns]
 1   Ticker                           39099 non-null  object        
 2   Close                            39099 non-null  float64       
 3   Open                             39099 non-null  float64       
 4   Volume                           39099 non-null  float64       
 5   CashAndCashEquivalents           2454 non-null   float64       
 6   CurrentAssets                    2308 non-null   float64       
 7   CurrentDebt                      1813 non-null   float64       
 8   CurrentLiabilities               2308 non-null   float64       
 9   FinancialsSource                 2460 non-null   object        
 10  LongTermDebt                     2303 non-null   float64  

* Se agregan los datos complementarios:

In [15]:
# Se agregan las columnas de info complementaria
df_final = pd.merge(
    df_merged, 
    df_supplementary, 
    on=['Ticker'],
    how='left' 
)
print("Columnas de info complementaria agregadas. Dimensiones:", df_final.shape)

Columnas de info complementaria agregadas. Dimensiones: (39099, 31)


* Limpieza final y almacenamiento de datos en fichero raw_data.parquet:

In [16]:
# Limpieza final
df_final_clean = limpieza_final(df_final)
print("Dimensiones del dataset final limpio:", df_final_clean.shape)

Dimensiones del dataset final limpio: (7027, 31)


In [17]:
df_final_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7027 entries, 60 to 39098
Data columns (total 31 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Date                             7027 non-null   datetime64[ns]
 1   Ticker                           7027 non-null   object        
 2   Close                            7027 non-null   float64       
 3   Open                             7027 non-null   float64       
 4   Volume                           7027 non-null   float64       
 5   CashAndCashEquivalents           7017 non-null   float64       
 6   CurrentAssets                    6586 non-null   float64       
 7   CurrentDebt                      5375 non-null   float64       
 8   CurrentLiabilities               6586 non-null   float64       
 9   FinancialsSource                 7027 non-null   object        
 10  LongTermDebt                     6610 non-null   float64       

In [18]:
# Guardar/actualizar fichero raw_data
df_guardado = guardar_raw_data(df_final_clean)

Extracción finalizada.
Datos guardados exitosamente en 'data\raw\raw_data.parquet'.


## Verificar datos guardados

In [19]:
df_guardado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39296 entries, 0 to 39295
Data columns (total 31 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ticker                           39296 non-null  object        
 1   Date                             39296 non-null  datetime64[ns]
 2   Close                            39296 non-null  float64       
 3   Open                             39296 non-null  float64       
 4   Volume                           39296 non-null  float64       
 5   CashAndCashEquivalents           39221 non-null  float64       
 6   CurrentAssets                    37218 non-null  float64       
 7   CurrentDebt                      30890 non-null  float64       
 8   CurrentLiabilities               37218 non-null  float64       
 9   FinancialsSource                 39296 non-null  object        
 10  LongTermDebt                     37050 non-null  float64  

In [20]:
print("Tickers sin datos:", len(sin_datos_set))
print("Tickers guardados:", len(df_guardado['Ticker'].unique().tolist()))    

Tickers sin datos: 3
Tickers guardados: 569
